# 2-6 NumPy 其他操作

本节学习数组中几种常用的查找、排列和筛选操作：

| 方法或写法 | 作用 |
|---|---|
| `np.argmin()` / `np.argmax()` | 查找最小值或最大值所在的索引 |
| `rng.shuffle()` | 原地打乱数组的顺序 |
| `np.sort()` | 返回排序后的数组 |
| `数组[布尔条件]` | 根据条件筛选元素 |

> 随机示例使用固定种子。`rng` 是通过 `np.random.default_rng()` 创建的随机数生成器。

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

## 1. 查找最值所在的索引

`min()`、`max()` 返回最值本身；`argmin()`、`argmax()` 返回最值对应的**索引**。

先创建一个包含 16 个随机数的一维数组：

In [ ]:
a = rng.random(16)
a

### 1.1 使用 `argmin()` 查找最小值的索引

In [ ]:
min_index = np.argmin(a)
min_index

将返回的索引用于数组取值，即可找到最小值。不要把某次运行看到的索引写死在代码里。

In [ ]:
a[min_index], np.min(a)

两种方式取得的数相同：前者先找到位置再取值，后者直接求最小值。

### 1.2 使用 `argmax()` 查找最大值的索引

In [ ]:
max_index = np.argmax(a)
max_index

In [ ]:
a[max_index], np.max(a)

**补充：** 如果最值出现多次，返回第一次出现的位置。多维数组不指定 `axis` 时，返回按默认顺序展开后的索引，不会直接返回“行、列”坐标。本节使用一维数组来说明基本用法。

## 2. 打乱数组顺序

`shuffle()` 会**直接修改原数组**，返回值为 `None`，因此不应该写成 `X = rng.shuffle(X)`。

`np.random.shuffle()` 和局部随机数生成器的 `rng.shuffle()` 都可以打乱数组顺序。本节使用固定种子初始化的局部随机数生成器，以便复现结果。

对二维数组，默认沿 `axis=0` 打乱：**改变各行的排列顺序，每一行内部的元素顺序保持不变**。

In [ ]:
X = np.arange(16).reshape(4, 4)
X

In [ ]:
rng.shuffle(X, axis=0)
X

再次执行打乱单元格会继续消耗随机数生成器的状态；从导入和初始化单元格开始顺序运行，才能复现整本笔记的结果。

## 3. 对数组排序

`np.sort()` 默认按升序排序，并**返回一个新数组**，不会修改传入的原数组。

| 写法 | 二维数组中的含义 |
|---|---|
| `np.sort(X, axis=0)` | 每一列分别从上到下排序 |
| `np.sort(X, axis=1)` | 每一行分别从左到右排序 |
| `np.sort(X)` | 默认沿最后一轴排序，二维数组中等同于 `axis=1` |

排序时的 `axis` 表示排序沿哪个方向进行，不是把该维度汇总掉。

### 3.1 沿第 0 轴排序

In [ ]:
np.sort(X, axis=0)

每一列分别升序排列。在当前例子中，结果恢复为原先从 0 到 15 的排列。

### 3.2 沿第 1 轴排序

In [ ]:
np.sort(X, axis=1)

之前只打乱了行的位置，各行内部本来就是升序，因此按行排序后，每行内部看起来没有变化。原数组 `X` 仍然保持打乱后的行顺序：

In [ ]:
X

换一个行内顺序也不同的数组，更容易看清两个轴的差别：

In [ ]:
unsorted = np.array([[9, 1, 5], [2, 8, 3]])
unsorted

In [ ]:
np.sort(unsorted, axis=0)

In [ ]:
np.sort(unsorted, axis=1)

> `np.sort(X)` 返回排序后的新数组；`X.sort()` 则会原地修改 `X`。学习时应先分清是否需要保留原始顺序。

## 4. 使用布尔索引选择元素

**布尔索引**用 `True` 和 `False` 表示是否保留对应位置的元素：

- `True`：保留该位置的元素。
- `False`：不保留该位置的元素。

### 4.1 手动指定布尔索引

`X[0, [True, False, False, True]]` 表示先选择第 1 行，再保留该行的第 1 个和第 4 个元素。

In [ ]:
X[0, [True, False, False, True]]

也可以把布尔列表保存为变量。这里列表长度必须与所选择行的元素个数一致，即 4 个。

In [ ]:
ind = [True, False, False, True]
X[0, ind]

### 4.2 根据数值范围生成布尔条件

筛选数组中位于 **`[3, 10)`** 的元素，即大于等于 3，同时小于 10。

- `X >= 3`：判断每个元素是否大于等于 3。
- `X < 10`：判断每个元素是否小于 10。
- `&`：要求两个条件同时满足。

**每个比较条件都要加括号**，数组的多个条件不能用 Python 的 `and` 直接连接。

In [ ]:
mask = (X >= 3) & (X < 10)
mask

`mask` 与 `X` 的形状相同。把它放入方括号，提取所有为 `True` 的位置：

In [ ]:
X[mask]

使用与原数组同形状的布尔数组筛选后，结果是一维数组。顺序取决于元素在当前 `X` 中的排列，因此筛选结果不一定按数值升序排列。

也可以不单独保存条件，直接写成：

In [ ]:
X[(X >= 3) & (X < 10)]

### 4.3 筛选能被 2 整除的元素

`%` 是取余运算。`X % 2 == 0` 表示每个元素除以 2 后余数是否为 0。

In [ ]:
X % 2 == 0

In [ ]:
X[X % 2 == 0]

## 5. 要点总结

- `argmin()` 和 `argmax()` 返回索引，`min()` 和 `max()` 返回对应的值。
- 用返回的索引读取元素，可以避免依赖某次随机结果中的固定位置。
- `shuffle()` 原地打乱数组；二维数组默认打乱行的位置。
- `np.sort()` 返回排序后的新数组，`axis=0` 对每列排序，`axis=1` 对每行排序。
- 布尔索引使用 `True` 保留元素，使用 `False` 排除元素。
- 多个数组条件使用 `&`（同时满足）或 `|`（至少满足一个）连接，每个比较条件都要加括号。
- `X[X % 2 == 0]` 可以筛选出数组中的偶数。